# Hyperparameter Tuning for the Photosensor ML Models

Companion notebook to `photosensor_ML_clean.ipynb` for *SCAPS-1D and Machine Learning-Assisted Optimization of a
Self-Powered ZnSe/Ca2CdAs2/MoS2 NIR PD*.

This notebook tunes the four tree-based regressors (DT, RF, GB and XGB) for both targets, responsivity $R$ and
log$_{10}D^*$. Linear regression has no hyperparameters and is not tuned. The tuned values found here are the ones
written into Section 3 of the main notebook, which then compares the five tuned models and selects one per target.

| Item | Setting |
|---|---|
| Data | Same preprocessing and bandgap-stratified, device-level 80/20 split as Sections 1–2 of the main notebook; only the 2,656 training devices are used |
| Search | `RandomizedSearchCV`, `random_state = 42`; DT 12 of 36 combinations, RF 10 of 16, GB 10 of 24, XGB 12 of 36 |
| Cross-validation | 3-fold `StratifiedGroupKFold` (shuffled, seed 42): grouped by `Structure_ID`, stratified by $E_g$ |
| Score | RMSE (`neg_root_mean_squared_error`); the candidate with the lowest mean CV RMSE is kept |
| Test devices | Not used in this notebook |

**Inputs:** `photosensor_dataset.xlsx` (SCAPS-1D output) in the working directory.
**Outputs:** `outputs/Tuned_Hyperparameters.csv`, `outputs/Tuned_Hyperparameters.json` and
`outputs/Tuning_All_Candidates.csv` (CV score of every candidate tried).


## 0. Setup

In [1]:
# If the dataset is on Google Drive, uncomment the next two lines and point DATA_PATH to it.
# from google.colab import drive
# drive.mount('/content/drive')

import os, sys, json, time, warnings
import numpy as np
import pandas as pd
import sklearn, xgboost
from IPython.display import display

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, StratifiedGroupKFold, RandomizedSearchCV, ParameterGrid
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)

DATA_PATH    = "photosensor_dataset.xlsx"
OUTPUT_DIR   = "outputs"
RANDOM_STATE = 42
N_CV_FOLDS   = 3

os.makedirs(OUTPUT_DIR, exist_ok=True)

RAW_COLUMNS = ["Eg", "Affinity", "t", "N_A", "N_t", "Wavelength",
               "Voc", "Jsc", "QE", "Responsivity", "Detectivity"]
STRUCTURE_COLUMNS = ["Eg", "Affinity", "t", "N_A", "N_t"]
FEATURES = ["Eg", "Affinity", "t", "N_A", "N_t", "Wavelength", "CBO", "VBO"]
TARGET_NAMES = {"Responsivity": "Responsivity", "log10(Detectivity)": "log_Detectivity"}

CHI_ZNSE = 4.09   # eV, electron affinity of ZnSe (window)
CHI_MOS2 = 3.80   # eV, electron affinity of MoS2 (BSF)
EG_MOS2  = 1.62   # eV, bandgap of MoS2

print(f"Python {sys.version.split()[0]} | scikit-learn {sklearn.__version__} | xgboost {xgboost.__version__}")
if tuple(int(v) for v in sklearn.__version__.split(".")[:2]) >= (1, 8):
    print("Note: scikit-learn >= 1.8 builds different StratifiedGroupKFold folds; results may differ from the shown outputs.")

Python 3.13.16 | scikit-learn 1.7.2 | xgboost 3.4.1


## 1. Data preparation

The steps are identical to Sections 1–2 of the main notebook: device IDs are assigned on the full dataset, rows with
missing targets or $D^* \le 0$ are removed, CBO and VBO are added, $D^*$ is log-transformed, and the device-level split is
stratified by $E_g$ with the same seed. Only the training devices are passed on.

In [2]:
df = pd.read_excel(DATA_PATH)
df.columns = df.columns.astype(str).str.strip()
for col in RAW_COLUMNS:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["Structure_ID"] = df.groupby(STRUCTURE_COLUMNS, sort=False).ngroup()

df_ml = df.dropna(subset=STRUCTURE_COLUMNS + ["Wavelength", "Responsivity", "Detectivity"])
df_ml = df_ml[df_ml["Detectivity"] > 0].copy()
df_ml["CBO"] = df_ml["Affinity"] - CHI_ZNSE
df_ml["VBO"] = CHI_MOS2 - df_ml["Affinity"] + EG_MOS2 - df_ml["Eg"]
df_ml["log_Detectivity"] = np.log10(df_ml["Detectivity"])

value_columns = FEATURES + ["Responsivity", "Detectivity", "log_Detectivity"]
df_clean = df_ml[["Structure_ID"] + value_columns].dropna()
df_clean = df_clean.loc[np.isfinite(df_clean[value_columns]).all(axis=1)].reset_index(drop=True)

structure_table = (df_clean[["Structure_ID"] + STRUCTURE_COLUMNS]
                   .drop_duplicates(subset="Structure_ID").reset_index(drop=True))
train_structures, test_structures = train_test_split(
    structure_table, test_size=0.20, random_state=RANDOM_STATE, stratify=structure_table["Eg"])

train_ids = set(train_structures["Structure_ID"])
test_ids  = set(test_structures["Structure_ID"])
assert not (train_ids & test_ids)

train_df = df_clean[df_clean["Structure_ID"].isin(train_ids)].reset_index(drop=True)
X_train  = train_df[FEATURES]
y_train  = {name: train_df[col] for name, col in TARGET_NAMES.items()}
groups   = train_df["Structure_ID"]
eg_label = train_df["Eg"].astype(str)

print(f"Clean dataset   : {len(df_clean):,} rows, {df_clean['Structure_ID'].nunique():,} devices")
print(f"Training subset : {len(train_df):,} rows, {len(train_ids):,} devices (used here)")
print(f"Test subset     : {len(df_clean) - len(train_df):,} rows, {len(test_ids):,} devices (not used here)")

Clean dataset   : 197,415 rows, 3,320 devices
Training subset : 157,928 rows, 2,656 devices (used here)
Test subset     : 39,487 rows, 664 devices (not used here)


## 2. Cross-validation folds

The folds are built once and shared by every search, so all models and candidates are scored on the same validation devices.

In [3]:
cv_splits = list(StratifiedGroupKFold(n_splits=N_CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
                 .split(X_train, eg_label, groups))

rows = []
for k, (tr, va) in enumerate(cv_splits, start=1):
    tr_dev, va_dev = set(groups.iloc[tr]), set(groups.iloc[va])
    assert not (tr_dev & va_dev), f"Device leakage in fold {k}"
    share = train_df.iloc[va].drop_duplicates("Structure_ID")["Eg"].value_counts().sort_index()
    rows.append({"Fold": k, "Train_devices": len(tr_dev), "Validation_devices": len(va_dev),
                 "Validation_rows": len(va), "Device_overlap": 0,
                 **{f"Val_devices_Eg={eg:.2f}": n for eg, n in share.items()}})

print("No device appears in both the training and validation part of any fold.")
display(pd.DataFrame(rows).set_index("Fold"))

No device appears in both the training and validation part of any fold.


,Train_devices,Validation_devices,Validation_rows,Device_overlap,Val_devices_Eg=0.62,Val_devices_Eg=0.79,Val_devices_Eg=0.82,Val_devices_Eg=0.95,Val_devices_Eg=1.00
Fold,,,,,,,,,
1,1771,885,52659,0,115,184,186,200,200
2,1771,885,52617,0,113,187,186,201,198
3,1770,886,52652,0,112,188,185,199,202


## 3. Models and search spaces

The base estimators use the same fixed settings as the main notebook; only the listed hyperparameters are searched.
`RandomizedSearchCV` draws the given number of distinct combinations from each grid.

In [4]:
BASE_MODELS = {
    "DT":  DecisionTreeRegressor(random_state=RANDOM_STATE),
    "RF":  RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    "GB":  GradientBoostingRegressor(random_state=RANDOM_STATE),
    "XGB": XGBRegressor(objective="reg:squarederror", eval_metric="rmse", tree_method="hist",
                        random_state=RANDOM_STATE, n_jobs=-1, verbosity=0),
}

SEARCH_SPACES = {
    "DT":  {"max_depth": [3, 5, 10, None], "min_samples_split": [2, 5, 10], "min_samples_leaf": [1, 2, 4]},
    "RF":  {"n_estimators": [100, 200], "max_depth": [3, 5, 10, None], "min_samples_split": [2, 5]},
    "GB":  {"n_estimators": [100, 200], "learning_rate": [0.01, 0.05, 0.10, 0.20], "max_depth": [3, 5, 10]},
    "XGB": {"n_estimators": [100, 200], "learning_rate": [0.01, 0.10, 0.20], "max_depth": [3, 6, 10],
            "subsample": [0.8, 1.0]},
}

N_ITER = {"DT": 12, "RF": 10, "GB": 10, "XGB": 12}

# RF and XGB use all cores internally, so their searches run one fit at a time
SEARCH_N_JOBS = {"DT": -1, "RF": 1, "GB": -1, "XGB": 1}

display(pd.DataFrame([
    {"Model": m,
     "Search space": "; ".join(f"{p} {{{', '.join(str(v) for v in vals)}}}" for p, vals in SEARCH_SPACES[m].items()),
     "Grid size": len(ParameterGrid(SEARCH_SPACES[m])),
     "Combinations tried": N_ITER[m]}
    for m in SEARCH_SPACES]).set_index("Model"))

,Search space,Grid size,Combinations tried
Model,,,
DT,"max_depth {3, 5, 10, None}; min_samples_split {2, 5, 10}; min_samples_leaf {1, 2, 4}",36,12
RF,"n_estimators {100, 200}; max_depth {3, 5, 10, None}; min_samples_split {2, 5}",16,10
GB,"n_estimators {100, 200}; learning_rate {0.01, 0.05, 0.1, 0.2}; max_depth {3, 5, 10}",24,10
XGB,"n_estimators {100, 200}; learning_rate {0.01, 0.1, 0.2}; max_depth {3, 6, 10}; subsample {0.8, 1.0}",36,12


## 4. Randomized search

Each candidate is fitted on two folds and scored on the third, three times. The search is run separately for each target.
Refitting on the full training set is not needed here (the main notebook does that), so `refit=False`.

In [5]:
best_records, all_candidates = [], []
run_start = time.time()

for target, y in y_train.items():
    print(f"\n=== Target: {target} ===")
    for name, estimator in BASE_MODELS.items():
        start = time.time()
        search = RandomizedSearchCV(
            estimator=estimator,
            param_distributions=SEARCH_SPACES[name],
            n_iter=N_ITER[name],
            scoring="neg_root_mean_squared_error",
            cv=cv_splits,
            refit=False,
            random_state=RANDOM_STATE,
            n_jobs=SEARCH_N_JOBS[name],
            error_score="raise",
        )
        search.fit(X_train, y)
        elapsed = time.time() - start

        res = pd.DataFrame(search.cv_results_)
        fold_cols = [f"split{k}_test_score" for k in range(N_CV_FOLDS)]
        res["CV_RMSE"] = -res["mean_test_score"]
        res["CV_RMSE_sd"] = (-res[fold_cols]).std(axis=1, ddof=0)
        for _, r in res.iterrows():
            all_candidates.append({"Target": target, "Model": name, "Rank": int(r["rank_test_score"]),
                                   "Parameters": json.dumps(r["params"]),
                                   "CV_RMSE": r["CV_RMSE"], "CV_RMSE_sd": r["CV_RMSE_sd"],
                                   **{f"Fold{k+1}_RMSE": -r[c] for k, c in enumerate(fold_cols)}})

        best = res.loc[search.best_index_]
        best_records.append({"Target": target, "Model": name, "Combinations_tried": len(res),
                             "Best_parameters": search.best_params_,
                             "CV_RMSE": best["CV_RMSE"], "CV_RMSE_sd": best["CV_RMSE_sd"],
                             "Search_time_s": round(elapsed, 1)})
        print(f"{name:<4} best CV RMSE = {best['CV_RMSE']:.8f} (sd {best['CV_RMSE_sd']:.2e})  "
              f"{search.best_params_}  [{elapsed:.0f} s]")

print(f"\nTotal search time: {(time.time() - run_start) / 60:.1f} min")


=== Target: Responsivity ===


DT   best CV RMSE = 0.00072350 (sd 1.64e-04)  {'min_samples_split': 5, 'min_samples_leaf': 2, 'max_depth': None}  [4 s]


RF   best CV RMSE = 0.00104937 (sd 4.05e-04)  {'n_estimators': 200, 'min_samples_split': 2, 'max_depth': None}  [277 s]


GB   best CV RMSE = 0.00088147 (sd 1.02e-04)  {'n_estimators': 200, 'max_depth': 10, 'learning_rate': 0.05}  [232 s]


XGB  best CV RMSE = 0.00080967 (sd 2.14e-04)  {'subsample': 0.8, 'n_estimators': 200, 'max_depth': 10, 'learning_rate': 0.2}  [14 s]

=== Target: log10(Detectivity) ===


DT   best CV RMSE = 0.20430047 (sd 1.12e-01)  {'min_samples_split': 2, 'min_samples_leaf': 2, 'max_depth': None}  [3 s]


RF   best CV RMSE = 0.25098768 (sd 1.16e-01)  {'n_estimators': 200, 'min_samples_split': 2, 'max_depth': None}  [271 s]


GB   best CV RMSE = 0.18356825 (sd 1.22e-01)  {'n_estimators': 100, 'max_depth': 10, 'learning_rate': 0.1}  [234 s]


XGB  best CV RMSE = 0.22693942 (sd 1.18e-01)  {'subsample': 1.0, 'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.2}  [18 s]

Total search time: 17.6 min


## 5. Tuned hyperparameters

Best candidate of every search, followed by the three best candidates of each search to show how close the alternatives were.
The full list of candidates is written to `outputs/Tuning_All_Candidates.csv`.

In [6]:
best_df = pd.DataFrame(best_records)
show = best_df.assign(Best_parameters=best_df["Best_parameters"].apply(
    lambda d: ", ".join(f"{k}={v}" for k, v in sorted(d.items()))))
display(show.set_index(["Target", "Model"]))

candidates_df = pd.DataFrame(all_candidates).sort_values(["Target", "Model", "Rank"]).reset_index(drop=True)
print("Three best candidates per search:")
display(candidates_df.groupby(["Target", "Model"], sort=False).head(3)
        [["Target", "Model", "Rank", "Parameters", "CV_RMSE", "CV_RMSE_sd"]].set_index(["Target", "Model"]))

best_df.assign(Best_parameters=best_df["Best_parameters"].apply(json.dumps)).to_csv(
    os.path.join(OUTPUT_DIR, "Tuned_Hyperparameters.csv"), index=False)
candidates_df.to_csv(os.path.join(OUTPUT_DIR, "Tuning_All_Candidates.csv"), index=False)
with open(os.path.join(OUTPUT_DIR, "Tuned_Hyperparameters.json"), "w") as f:
    json.dump({t: {r["Model"]: r["Best_parameters"] for r in best_records if r["Target"] == t} for t in y_train},
              f, indent=2)
print("Saved: Tuned_Hyperparameters.csv, Tuned_Hyperparameters.json, Tuning_All_Candidates.csv")

Combinations_tried                                                   Best_parameters   CV_RMSE  CV_RMSE_sd  Search_time_s
Target             Model                                                                                                                           
Responsivity       DT                     12           max_depth=None, min_samples_leaf=2, min_samples_split=5  0.000724    0.000164            4.5
                   RF                     10             max_depth=None, min_samples_split=2, n_estimators=200  0.001049    0.000405          277.0
                   GB                     10                learning_rate=0.05, max_depth=10, n_estimators=200  0.000881    0.000102          232.1
                   XGB                    12  learning_rate=0.2, max_depth=10, n_estimators=200, subsample=0.8  0.000810    0.000214           14.3
log10(Detectivity) DT                     12           max_depth=None, min_samples_leaf=2, min_samples_split=2  0.204300    0.111858            3.2
                   RF                     10             max_depth=None, min_samples_split=2, n_estimators=200  0.250988    0.116104          271.4
                   GB                     10                 learning_rate=0.1, max_depth=10, n_estimators=100  0.183568    0.122314          234.0
                   XGB                    12   learning_rate=0.2, max_depth=6, n_estimators=200, subsample=1.0  0.226939    0.117575           18.0

Three best candidates per search:


Rank                                                                      Parameters   CV_RMSE  CV_RMSE_sd
Target             Model                                                                                                            
Responsivity       DT        1              {"min_samples_split": 5, "min_samples_leaf": 2, "max_depth": null}  0.000724    0.000164
                   DT        2              {"min_samples_split": 2, "min_samples_leaf": 2, "max_depth": null}  0.000726    0.000162
                   DT        3              {"min_samples_split": 5, "min_samples_leaf": 4, "max_depth": null}  0.003782    0.003844
                   GB        1                   {"n_estimators": 200, "max_depth": 10, "learning_rate": 0.05}  0.000881    0.000102
                   GB        2                    {"n_estimators": 100, "max_depth": 10, "learning_rate": 0.1}  0.000910    0.000085
                   GB        3                     {"n_estimators": 200, "max_depth": 5, "learning_rate": 0.2}  0.004322    0.000234
                   RF        1                {"n_estimators": 200, "min_samples_split": 2, "max_depth": null}  0.001049    0.000405
                   RF        2                {"n_estimators": 200, "min_samples_split": 5, "max_depth": null}  0.001155    0.000433
                   RF        3                {"n_estimators": 100, "min_samples_split": 5, "max_depth": null}  0.001278    0.000587
                   XGB       1  {"subsample": 0.8, "n_estimators": 200, "max_depth": 10, "learning_rate": 0.2}  0.000810    0.000214
                   XGB       2  {"subsample": 1.0, "n_estimators": 100, "max_depth": 10, "learning_rate": 0.1}  0.000968    0.000161
                   XGB       3  {"subsample": 1.0, "n_estimators": 200, "max_depth": 10, "learning_rate": 0.2}  0.000978    0.000113
log10(Detectivity) DT        1              {"min_samples_split": 2, "min_samples_leaf": 2, "max_depth": null}  0.204300    0.111858
                   DT        2              {"min_samples_split": 5, "min_samples_leaf": 4, "max_depth": null}  0.220355    0.106963
                   DT        3              {"min_samples_split": 5, "min_samples_leaf": 2, "max_depth": null}  0.261587    0.100391
                   GB        1                    {"n_estimators": 100, "max_depth": 10, "learning_rate": 0.1}  0.183568    0.122314
                   GB        2                   {"n_estimators": 200, "max_depth": 10, "learning_rate": 0.05}  0.209174    0.111407
                   GB        3                     {"n_estimators": 200, "max_depth": 5, "learning_rate": 0.2}  0.230401    0.116345
                   RF        1                {"n_estimators": 200, "min_samples_split": 2, "max_depth": null}  0.250988    0.116104
                   RF        2                {"n_estimators": 200, "min_samples_split": 5, "max_depth": null}  0.251997    0.116768
                   RF        3                {"n_estimators": 100, "min_samples_split": 5, "max_depth": null}  0.258066    0.119151
                   XGB       1   {"subsample": 1.0, "n_estimators": 200, "max_depth": 6, "learning_rate": 0.2}  0.226939    0.117575
                   XGB       2  {"subsample": 1.0, "n_estimators": 100, "max_depth": 10, "learning_rate": 0.1}  0.234517    0.112305
                   XGB       3  {"subsample": 1.0, "n_estimators": 200, "max_depth": 10, "learning_rate": 0.2}  0.236887    0.113884

Saved: Tuned_Hyperparameters.csv, Tuned_Hyperparameters.json, Tuning_All_Candidates.csv
